In [5]:
import numpy as np
import matplotlib.pyplot as plt

## Scenario 1

### Tasks

1. Implement ideal sinc reconstruction.
2. Determine the sampling period from `sample_times`.
3. Add a function

```python
def sampling_status(fs):
    ...
```

that returns `"safe"` if Nyquist is satisfied and `"aliased"` otherwise.
4. For

```text
fs = 1200 Hz
fs = 700 Hz
```

predict what will happen.
5. Modify the reconstruction code so that it works for complex-valued samples too.
6. The engineer says:

> “At \(700\) Hz the reconstructed curve still looks smooth, therefore there is no aliasing.”

Explain the exact conceptual error.


In [8]:
def vibration_signal(t):
    return (
        1.5 * np.cos(2*np.pi*120*t)
        + 0.8 * np.sin(2*np.pi*310*t)
        + 0.3 * np.cos(2*np.pi*420*t)
    )

def sample_signal(signal, fs, duration):
    ts = np.arange(0, duration, 1/fs)
    xs = signal(ts)
    return ts, xs

def reconstruct(samples, sample_times, output_times):
    samples = np.asarray(samples)
    sample_times = np.asarray(sample_times)
    output_times = np.asarray(output_times)

    if len(sample_times) < 2:
        raise ValueError("Need at least two samples")

    T = sample_times[1] - sample_times[0]

    kernel = np.sinc(
        (output_times[:, None] - sample_times[None, :]) / T
    )

    return kernel @ samples


def sampling_status(fs):
    f_max = 420.0

    if fs > 2 * f_max:
        return "safe"

    return "aliased"

def run_monitor(fs):
    duration = 0.05

    ts, xs = sample_signal(vibration_signal, fs, duration)

    tdense = np.linspace(0, duration, 6000)
    xr = reconstruct(xs, ts, tdense)

    return tdense, xr

### Explanation for the reconstrut function:


#### Kernel Matrix

```python

    kernel = np.sinc(
        (output_times[:, None] - sample_times[None, :]) / T
    )
```

The kernel matrix is an **influence table**. It measures how much every original sample point influences every single query point you want to calculate.

---

**1. The Broadcasting Trick (`[:, None]` vs `[None, :]`)**

Assume you have 2 original samples at $t = [0, 1]$ and you want to reconstruct the signal at 3 query times $t = [0.0, 0.5, 1.0]$ with $T = 1$:

* `output_times[:, None]` turns your 3 query points into a **column vector** of shape `(3, 1)`:

$$\begin{bmatrix} 0.0 \\ 0.5 \\ 1.0 \end{bmatrix}$$


* `sample_times[None, :]` turns your 2 sample points into a **row vector** of shape `(1, 2)`:

$$\begin{bmatrix} 0 & 1 \end{bmatrix}$$



When NumPy subtracts them, it stretches (broadcasts) both into a **$3 \times 2$ grid of time distances**:

$$\text{Time Differences} = \begin{bmatrix} 0.0 - 0 & 0.0 - 1 \\ 0.5 - 0 & 0.5 - 1 \\ 1.0 - 0 & 1.0 - 1 \end{bmatrix} = \begin{bmatrix} 0.0 & -1.0 \\ 0.5 & -0.5 \\ 1.0 & 0.0 \end{bmatrix}$$

* **Row index ($i$):** Which **output time** you are calculating.
* **Column index ($j$):** Which **original sample** is exerting influence.

---

**2. Passing Distances Through `np.sinc**`

Dividing by $T$ and applying `np.sinc` converts those raw time differences into **interpolation weights**:

$$\text{kernel} = \begin{bmatrix}  \text{sinc}(0.0) & \text{sinc}(-1.0) \\  \text{sinc}(0.5) & \text{sinc}(-0.5) \\  \text{sinc}(1.0) & \text{sinc}(0.0)  \end{bmatrix}$$

Since $\text{sinc}(0) = 1$ and $\text{sinc}(\text{non-zero integer}) = 0$:

$$\text{kernel} = \begin{bmatrix}  1.0 & 0.0 \\  0.637 & 0.637 \\  0.0 & 1.0  \end{bmatrix}$$

* **Row 0 ($t = 0.0$):** Sample 0 has $100\%$ weight (`1.0`); Sample 1 has $0\%$ weight (`0.0`).
* **Row 1 ($t = 0.5$):** Directly in the middle, so both samples contribute equal weight (`0.637`).
* **Row 2 ($t = 1.0$):** Sample 1 has $100\%$ weight (`1.0`); Sample 0 has $0\%$ weight (`0.0`).

---

**3. The Matrix Multiplication (`kernel @ samples`)**

The `@` operator takes the dot product of each row of weights with your sample values:

$$\begin{bmatrix}  1.0 & 0.0 \\  0.637 & 0.637 \\  0.0 & 1.0  \end{bmatrix}  \begin{bmatrix}  \text{Sample}_0 \\  \text{Sample}_1  \end{bmatrix}  =  \begin{bmatrix}  (1.0 \times \text{Sample}_0) + (0.0 \times \text{Sample}_1) \\  (0.637 \times \text{Sample}_0) + (0.637 \times \text{Sample}_1) \\  (0.0 \times \text{Sample}_0) + (1.0 \times \text{Sample}_1)  \end{bmatrix}$$

Instead of running nested Python `for` loops to place a centered sinc pulse over every sample point and add them up, this single matrix operation computes the entire continuous reconstruction in parallel.


